In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfTransformer

: 

In [ ]:

data = pd.read_csv("../data/snsdata.csv")
data.head()

In [ ]:
# Variables de caracterización:
# - gradyear: año de graduación;
# - gender: género;
# - age: edad;
# - friends`: número de amigos.

data.info()

In [ ]:
# Calidad de datos:

data[["gradyear", "gender", "age", "friends"]].describe(include="all")

In [ ]:
# Diagnóstico de la variable edad:

print("Edades faltantes:", data["age"].isna().sum())
print("Edad mínima:", data["age"].min())
print("Edad máxima:", data["age"].max())

In [ ]:
# Remoción de datos con edades fuera del rango de 13 a 20 años (inclusive):, la tabla será sobre conteo de la media, la mediana 

data["age_clean"] = data["age"].where(data["age"].between(13, 20, inclusive="left"))

age_by_gradyear = (
    data.groupby("gradyear")["age_clean"].agg(["count", "mean", "median"]).round(3)
)

age_by_gradyear

In [ ]:
# Imputación de edades faltantes con la mediana de edad por año de graduación:

age_medians = data.groupby("gradyear")["age_clean"].median()

data["age_imputed"] = data["age_clean"].fillna(data["gradyear"].map(age_medians))

print("Edades faltantes antes de imputar:", data["age_clean"].isna().sum())
print("Edades faltantes después de imputar:", data["age_imputed"].isna().sum())

data[["gradyear", "age", "age_clean", "age_imputed"]].head(10)

In [ ]:
# Variables usadas para construir los segmentos

profile_columns = ["gradyear", "gender", "age_imputed", "friends"]
interest_columns = data.columns[4:40].tolist()

print("Variables de perfil:", profile_columns)
print("Número de variables de interés:", len(interest_columns))
interest_columns


In [ ]:
# Segmentación con K-Means

N_CLUSTERS = 5
RANDOM_STATE = 42

kmeans = KMeans(
    n_clusters=N_CLUSTERS,
    n_init=30,
    random_state=RANDOM_STATE,
)

X_counts = data[interest_columns]

tfidf = TfidfTransformer()
X = tfidf.fit_transform(X_counts)


clusters = kmeans.fit_predict(X)

data["cluster"] = kmeans.fit_predict(X)

cluster_sizes = data["cluster"].value_counts().sort_index().rename("n").to_frame()

cluster_sizes["percentage"] = (100 * cluster_sizes["n"] / len(data)).round(2)

cluster_sizes

In [ ]:
ax = cluster_sizes["n"].plot(
    kind="bar",
    figsize=(8, 4),
    title="Tamaño de los segmentos",
)

ax.set_xlabel("Cluster")
ax.set_ylabel("Número de personas")
plt.tight_layout()
plt.show()

In [ ]:
# ¿Qué caracteriza a cada cluster?
#
#   - valores positivos: el cluster menciona ese interés más que el promedio;
#   - valores negativos: el cluster lo menciona menos que el promedio;
#   - valores cercanos a cero: comportamiento similar al promedio general.

centers = pd.DataFrame(
    kmeans.cluster_centers_,
    columns=interest_columns,
)


def top_interests(cluster_id, n=8):
    return (
        centers.loc[cluster_id]
        .sort_values(ascending=False)
        .head(n)
        .rename("standardized_score")
        .to_frame()
    )


for cluster_id in range(N_CLUSTERS):
    print(f"\nCLUSTER {cluster_id}")
    display(top_interests(cluster_id))

In [ ]:
top_features = []

for cluster_id in range(N_CLUSTERS):
    top = centers.loc[cluster_id].sort_values(ascending=False).head(6)
    for feature, score in top.items():
        top_features.append(
            {
                "cluster": cluster_id,
                "interest": feature,
                "score": score,
            }
        )

top_features = pd.DataFrame(top_features)

pivot_top = top_features.pivot(
    index="interest", columns="cluster", values="score"
).fillna(0)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(pivot_top.values, aspect="auto")

ax.set_xticks(range(N_CLUSTERS))
ax.set_xticklabels([f"Cluster {i}" for i in range(N_CLUSTERS)])
ax.set_yticks(range(len(pivot_top.index)))
ax.set_yticklabels(pivot_top.index)

ax.set_title("Intereses más distintivos de los clusters")
fig.colorbar(im, ax=ax, label="Centro estandarizado")

plt.tight_layout()
plt.show()

In [ ]:
# Perfil demográfico y social de los segmentos

data["gender_profile"] = data["gender"].fillna("Unknown")

cluster_profile = (
    data.groupby("cluster")
    .agg(
        n=("cluster", "size"),
        age_mean=("age_imputed", "mean"),
        age_median=("age_imputed", "median"),
        friends_mean=("friends", "mean"),
        friends_median=("friends", "median"),
    )
    .round(2)
)

cluster_profile

In [ ]:
gender_profile = (
    pd.crosstab(
        data["cluster"],
        data["gender_profile"],
        normalize="index",
    )
    .mul(100)
    .round(1)
)

gender_profile

In [ ]:
gradyear_profile = (
    pd.crosstab(
        data["cluster"],
        data["gradyear"],
        normalize="index",
    )
    .mul(100)
    .round(1)
)

gradyear_profile

In [ ]:
# Interpretación de los clusters
#
# En la ejecución de referencia con `random_state=42`, los perfiles dominantes son aproximadamente:
#
#  Cluster 0 — Vida social y comportamientos de riesgo
#    Los términos más distintivos incluyen
#
#                    `kissed`, `sex`, `drugs`, `hair`, `drunk`, `blonde` y `die`.
#
#    Interpretación: perfiles con conversación más intensa alrededor de relaciones, apariencia, vida
#    social y comportamientos de riesgo.
#
#  Cluster 1 — Moda, marcas y compras
#    Destacan
#          `hollister`, `abercrombie`, `shopping`, `mall`, `clothes`, `cheerleading`, `hair` y `cute`.
#
#    Interpretación: segmento claramente orientado a moda, marcas, centros comerciales y apariencia.
#
#
#  Cluster 2 — Baja intensidad de intereses específicos
#    Es el cluster más grande y sus centroides se encuentran, en general, cerca o por debajo del
#    promedio en las variables de interés.
#
#    Interpretación: usuarios con menor intensidad de señal en las categorías analizadas o con
#    intereses menos especializados.
#
#  Cluster 3 — Intereses sociales, deportivos y religiosos
#    Sobresalen
#        `church`, `god`, `shopping`, `basketball`, `jesus`, `cute`, `dance` y `football`.
#
#    Interpretación: segmento amplio con una combinación de actividad social, deportes, compras
#    y referencias religiosas.
#
#  Cluster 4 — Banda y música
#    Los términos más distintivos son
#                                `marching`, `band` y `music`.
#
#    Interpretación: nicho pequeño y muy especializado alrededor de bandas escolares y
#    actividades musicales.
#
# NOTA: Los nombres de los segmentos son **etiquetas analíticas construidas después del clustering.
# El algoritmo no conoce ni genera estos nombres: estos surgen de la interpretación de los perfiles.

In [ ]:
# Tabla resumen para una decisión de marketing

segment_names = {
    0: "Vida social y comportamientos de riesgo",
    1: "Moda, marcas y compras",
    2: "Baja intensidad de intereses específicos",
    3: "Social, deportes y religión",
    4: "Banda y música",
}

summary = cluster_profile.copy()
summary["segment"] = summary.index.map(segment_names)

summary = summary[["segment", "n", "age_mean", "friends_median"]]

summary

In [ ]:
# Almacena el resultado

output = data.copy()
output["segment"] = output["cluster"].map(segment_names)

output.to_csv(
    "../submission/segmented.csv",
    index=False,
)